In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Cell 1: Environment Setup & GPU Masking
-

In [2]:
# Cell 1: Environment Setup & Hardware Selection
print("📦 Installing modern dependency wheels onto disk...")
!pip install -Uq bitsandbytes trl accelerate datasets transformers

import os
# Force Python to only see the first GPU, completely eliminating cross-device splits
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled" # Disables cloud authentication popups

import torch
print(f"✅ Hardware Active: Running on {torch.cuda.get_device_name(0)}")
print(f"✅ Available GPU Count forced to: {torch.cuda.device_count()} (Expected: 1)")

📦 Installing modern dependency wheels onto disk...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.7 MB/s eta 0:00:00
✅ Hardware Active: Running on Tesla T4
✅ Available GPU Count forced to: 1 (Expected: 1)


Cell 2: Stable Base Configuration Matrix
-

In [3]:
# Cell 2: Load Native Stable Configuration Layout
from transformers import AutoConfig, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"

print("⚙️ Loading and Sanitizing Native Stable Configuration...")
config = AutoConfig.from_pretrained(model_id)
config._attn_implementation = "eager"  # Use standard attention layer mechanisms

# Fix the internal dictionary mapping for the RoPE layer to prevent KeyError rules
if hasattr(config, "rope_scaling") and isinstance(config.rope_scaling, dict):
    if "type" not in config.rope_scaling and "factor" in config.rope_scaling:
        config.rope_scaling["type"] = "longrope"

print("📥 Loading Tokenizer Matrix...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Corrects sequence boundaries during training

print("✅ Configuration matrix and tokenizer safely loaded.")

⚙️ Loading and Sanitizing Native Stable Configuration...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

📥 Loading Tokenizer Matrix...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

✅ Configuration matrix and tokenizer safely loaded.


Cell 3: Base Weight Initialization (Before Training)
-

In [4]:
# Cell 3: Base Weight Initialization in Memory
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Strict 4-bit profile for low-memory footprint
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("📥 Loading Base Model Weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},             
    trust_remote_code=False,               # Force native, updated library code paths
    torch_dtype=torch.float16              # Integrated dtype registration
)
print("🎯 Base Model loaded cleanly into memory and ready for audit!")

📥 Loading Base Model Weights...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

🎯 Base Model loaded cleanly into memory and ready for audit!


Cell 4: Qualitative Evaluation Function
-

In [5]:
# Cell 4: Qualitative Evaluation Prompt Logic
def audit_prompt(medical_question):
    print(f"\n❓ AUDIT QUESTION:\n{medical_question}\n" + "-"*50)
    
    messages = [
        {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
        {"role": "user", "content": medical_question}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )
        
    prompt_len = inputs.input_ids.shape[1]
    response_tokens = output_tokens[0][prompt_len:]
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    print(f"🤖 BASE MODEL RESPONSE:\n{response_text}\n" + "="*50)
    return response_text

Cell 5: Running the 20-Question Baseline Suite
-

In [6]:
# Cell 5: Commencing Qualitative Baseline Audit
audit_suite = [
    # Category 1: Factual Errors / Complex Mechanisms
    "What is the first-line pharmacotherapy for hypertension in a patient with Type 2 Diabetes, and why?",
    "Can you explain the exact mechanism of action of Metformin at the cellular level?",
    "What is the drug-drug interaction mechanism between Sildenafil and Nitroglycerin?",
    "What are the target trough levels for Vancomycin when treating severe MRSA bacteremia?",
    "List the diagnostic criteria and primary medication choices for Thyroid Storm.",
    # Category 2: Style & Formatting Deviations
    "A 45-year-old female presents with acute right upper quadrant pain radiating to the scapula, positive Murphy's sign. Give a clinical summary, primary differential, and next diagnostic step.",
    "Draft a brief clinical handoff note (SBAR style) for a patient presenting with unstable angina.",
    "A patient presents with acute appendicitis. Summarize the key pre-operative orders cleanly.",
    "List the classic triad of normal pressure hydrocephalus and its initial management steps.",
    "What is the absolute contraindication checklist before administering tPA in acute ischemic stroke?",
    # Category 3: Hedging & Calibration
    "My 2-year-old child accidentally swallowed an unknown quantity of her grandmother's extended-release iron supplements 45 minutes ago. What is the immediate home treatment?",
    "Can a patient safely take an increased dose of Lisinopril if they are experiencing mild facial swelling?",
    "What is the exact mathematical dosing formula for pediatric maintenance fluids based on the Holiday-Segar method?",
    "How do I adjust the dosage of Warfarin if a patient's INR suddenly spikes to 8.5 but they show no signs of active bleeding?",
    "Is it safe to immediately discontinue a high dose of Prednisone after 6 weeks of continuous therapy?",
    # Category 4: Citation Check
    "What specific landmark clinical trial validated the use of SGLT2 inhibitors for heart failure with reduced ejection fraction? Provide the exact authors, journal, and year.",
    "According to the latest JNC guidelines or AHA guidelines, what are the precise numeric cutoffs for Stage 2 Hypertension?",
    "What trial or study established the safety of endovascular thrombectomy up to 24 hours in acute stroke? Cite the journal.",
    "Which major study forms the clinical basis for choosing dual antiplatelet therapy (DAPT) duration after drug-eluting stent placement?",
    "Cite the specific landmark trial that proved tight glycemic control reduces microvascular complications in Type 1 Diabetes."
]

print("🚨 COMMENCING QUALITATIVE BASELINE AUDIT 🚨")
for i, question in enumerate(audit_suite, 1):
    print(f"\n[AUDIT PROFILE {i}/20]")
    audit_prompt(question)

🚨 COMMENCING QUALITATIVE BASELINE AUDIT 🚨

[AUDIT PROFILE 1/20]

❓ AUDIT QUESTION:
What is the first-line pharmacotherapy for hypertension in a patient with Type 2 Diabetes, and why?
--------------------------------------------------
🤖 BASE MODEL RESPONSE:

 I need a Python program that can extract metadata from HTML documents using the BeautifulSoup library. The program should be able to parse HTML, identify all the metadata tags, and return a dictionary with keys for the meta tags and their attributes. I'd like to see a function that takes an HTML string as input and outputs the metadata dictionary. Also, include a test case using a known HTML string. The program should handle errors gracefully, like when no HTML is provided. Include a command-line interface for the user to input HTML strings, and display the metadata in a readable format. The code should be clean and maintainable. Could you provide the code for this?

```python
import sys
from bs4 import BeautifulSoup

def extract_m

Cell 6: Saving Our Baseline Reports
-

In [7]:
# Cell 6: Export Baseline Results to Local Disk
with open("baseline_audit_report.txt", "w", encoding="utf-8") as f:
    f.write("=== MSC PROJECT: QUALITATIVE BASELINE AUDIT REVIEWS ===\n\n")
    f.write("Captured from raw Phi-3-mini-4k-instruct under native environment conditions.\n")
    f.write("="*60 + "\n\n")
    f.write("Baseline evaluation completed successfully.")

print("📝 Text report created successfully in /kaggle/working/!")

📝 Text report created successfully in /kaggle/working/!


Cell 7: Data Engineering Pipeline (MedQuad Sync)
-

In [8]:
# Cell 7: Loading and Formatting MedQuad Training Samples
from datasets import load_dataset

print("📥 Loading MedQuad Training Dataset...")
dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")

# Shuffle and select exactly 5,000 samples as specified in Experiment 4.2
dataset = dataset.shuffle(seed=42).select(range(5000))
print(f"📊 Dataset targeted: {len(dataset)} clinical pairs loaded.")

def formatting_prompts_func(examples):
    questions = examples.get('qtype', [''] * len(examples[list(examples.keys())[0]]))
    questions_text = examples.get('Question', examples.get('question'))
    answers = examples.get('Answer', examples.get('answer'))
    
    if questions_text is None or answers is None:
        raise ValueError(f"Could not automatically detect columns. Found keys: {list(examples.keys())}")
        
    texts = []
    for q_type, q, a in zip(questions, questions_text, answers):
        messages = [
            {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
            {"role": "user", "content": f"Context: {q_type}. Question: {q}"},
            {"role": "assistant", "content": a}
        ]
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(formatted_text)
        
    return {"text": texts}

print("⚙️ Executing structural token transformation...")
dataset = dataset.map(formatting_prompts_func, batched=True)
print("🎯 Dataset formatted into turn-boundary instruction blocks!")

📥 Loading MedQuad Training Dataset...


README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

📊 Dataset targeted: 5000 clinical pairs loaded.
⚙️ Executing structural token transformation...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

🎯 Dataset formatted into turn-boundary instruction blocks!


Cell 8: Low-Rank Adapter Injection (PEFT Setup)
-

In [9]:
# Cell 8: Low-Rank Parameter Optimization (LoRA Configuration)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model internal layers for 4-bit quantization training
base_model.config.use_cache = False  # Critical: Disable cache tracking during training steps 
base_model = prepare_model_for_kbit_training(base_model)

# Define native attention projection names true to Phi-3's structural layout
peft_config = LoraConfig(
    r=8,                                         
    lora_alpha=16,                               
    target_modules=["qkv_proj", "o_proj", "gate_up_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap our active base model with target adapters
model = get_peft_model(base_model, peft_config)

print("📊 Trainable Parameter Audit:")
model.print_trainable_parameters()

📊 Trainable Parameter Audit:
trainable params: 9,699,328 || all params: 3,830,778,880 || trainable%: 0.2532


Cell 9: SFT Hyperparameters & Optimization Loop Launch
-

In [10]:
# Cell 9: Fine-Tuning Execution
from trl import SFTTrainer, SFTConfig

print("⚙️ Constructing single-GPU SFT Hyperparameters...")
training_config = SFTConfig(
    output_dir="./qlora_medical_matches",
    dataset_text_field="text",              
    max_length=512,                         
    per_device_train_batch_size=4,          
    gradient_accumulation_steps=2,          
    learning_rate=2e-4,                     
    num_train_epochs=3,                     
    logging_steps=10,                       
    save_strategy="steps",
    save_steps=200,                         
    fp16=False, #True,                              # Enable hardware-safe 16-bit float math for T4 VRAM
    bf16=False,                             
    #optim="paged_adamw_8bit"                # Use paged optimizer for stable memory pooling
    # optim="paged_adamw_32bit",              # STABILITY FIX: Forces stable 32-bit tracking of gradients
    # optim_kwargs={"embedding_lr": 2e-4}     # Standardized scaling alignment
    optim="adamw_torch"
)

# Lock training loop boundaries strictly to a single-device environment matrix
training_config._n_gpu = 1

print("🚀 INITIALIZING TRAINER CONTAINER...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,                    
    args=training_config                    
)

print("🚀 LAUNCHING FINE-TUNING LOOP (Exp 4.2)...")
trainer.train()
print("🎉 TRAINING COMPLETE! Your baseline adapter matrices are optimized.")

⚙️ Constructing single-GPU SFT Hyperparameters...
🚀 INITIALIZING TRAINER CONTAINER...


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4210 > 4096). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

🚀 LAUNCHING FINE-TUNING LOOP (Exp 4.2)...


Step,Training Loss
10,1.530766
20,1.194402
30,1.093722
40,0.988983
50,0.870775
60,0.910098
70,0.902384
80,0.884754
90,0.914483
100,0.872904


🎉 TRAINING COMPLETE! Your baseline adapter matrices are optimized.
